# Online DSpark Training for Llama-3.2-1B-Instruct (single A100)

This notebook trains a [DSpark](https://docs.vllm.ai/projects/speculators/en/latest/user_guide/algorithms/dspark/)
speculator (the *draft* model) for `unsloth/Llama-3.2-1B-Instruct` (the *verifier*, or target).
The drafter proposes a block of tokens at a time; the verifier checks the whole block in a single
forward pass and keeps the longest correct prefix. Since every kept token is one the verifier itself
would have produced, decoding is faster but the output distribution is unchanged.

`speculators` implements several draft architectures (Eagle-3, P-EAGLE, DFlash, DFlash2, DSpark,
MTP). **DSpark** builds on DFlash's block-based diffusion decoder and adds a Markov logit bias plus
per-position confidence heads, which is what lets it propose a whole block of `--block-size` tokens
at once rather than one token at a time.

**Online training.** A drafter conditions on the verifier's *hidden states*, which have to come from
somewhere. `speculators` offers three modes:

| Mode | Hidden states | Trade-off |
| --- | --- | --- |
| offline | all pre-generated to disk before training | fastest epochs, enormous disk footprint |
| hybrid | generated during epoch 0, cached and reused | middle ground |
| **online** (used here) | fetched from a live vLLM server per batch, then discarded | near-zero disk, needs the verifier resident throughout |

**One GPU, two jobs.** This is the key difference from the 2x T4 Kaggle notebook, where the verifier
got its own card. Here a single A100 hosts both: vLLM is launched with
`--gpu-memory-utilization 0.25`, reserving a quarter of the card for the verifier and its KV cache,
and training runs in the remaining ~75% on the same device (`CUDA_VISIBLE_DEVICES=0` for both). The
verifier is only 1B parameters, so a quarter of an 80GB card is ample; the headroom buys a much
larger training configuration than the T4 run could fit.

**A bigger drafter.** The extra memory and compute are spent on capacity and context rather than on
more data passes: **3** layers instead of 2, `--block-size 8` instead of 4, `--total-seq-len 8192`
instead of 4096, and `--max-anchors 1024` instead of 256. A block of 8 raises the acceptance ceiling
but makes the later slots harder to hit, so this is a genuinely different trade-off, not just a
scaled-up version of the T4 run.

**Inputs and outputs.** The training data is the on-policy JSONL produced by
`data-preparation.ipynb` and published to
[`yosefw/magpie-llama-3.2-1b-instruct`](https://huggingface.co/datasets/yosefw/magpie-llama-3.2-1b-instruct).
The output is a self-contained drafter checkpoint pushed to
`yosefw/Llama-3.2-1B-Instruct-DSpark-OPB-3L`.

**Environment.** Written for a Colab runtime with one A100 and a Hugging Face token with write scope
in Colab Secrets. The `/content/...` paths throughout are Colab's working directory - on Kaggle they
do not exist and need repointing at `/kaggle/working/`.

**Reference.**
[Speculators training tutorial](https://docs.vllm.ai/projects/speculators/en/latest/user_guide/tutorials/train/)

## Step 0 - Set up the environment

Everything in this notebook is driven by scripts that live in the `speculators` repository under
`scripts/` (`prepare_data.py`, `launch_vllm.py`, `train.py`, `evaluate/`). Those are not shipped in
the published wheel, so the repository is cloned rather than `pip install speculators`. The `%cd`
below is persistent for the rest of the notebook: every path from here on is relative to the
checkout at `/content/speculators`.

In [ ]:
!git clone https://github.com/vllm-project/speculators.git

In [ ]:
%cd speculators

### Install vLLM and speculators

| Command | Why |
| --- | --- |
| `uv venv speculators_venv` | Creates the venv the tutorial names, kept for parity with the docs. |
| `uv pip install "vllm>=0.22.0"` | Serves the verifier, supplies hidden states, and later serves the trained drafter. |
| `uv pip install -e .` | Installs `speculators` from the clone, so the in-tree scripts are importable. |

The upstream tutorial keeps training and serving in two separate venvs to stop the training stack
from fighting vLLM over torch versions. Here they share one, which is what online training needs
anyway - `train.py` talks to the server over HTTP, so only one torch install is ever active.

Caveat: each `!` line runs in its own shell, so `source speculators_venv/bin/activate` does not carry
over to the next line or cell. The packages land in the runtime's own Python environment, which is
what the later cells actually run against. The venv is therefore nominal.

In [ ]:
# Speculators venv (for data prep and training)
! uv venv speculators_venv
! source speculators_venv/bin/activate
! uv pip install "vllm>=0.22.0"
! uv pip install -e .

### Fix up the preinstalled packages

The Colab image ships a `torchaudio` build pinned to its own torch version, which conflicts with the
torch vLLM pulls in and breaks the import chain; nothing here needs audio, so it is removed.
`datasets` is then upgraded because the loaders used in the next step need a newer version than the
image provides - `--break-system-packages` is required to overwrite the distribution-managed copy.

Colab usually asks for a runtime restart after this. Restarting is fine - the clone and the `%cd`
survive, but re-run the `%cd speculators` cell afterwards so the working directory is right again.

In [ ]:
! pip uninstall -y torchaudio
! pip install -Uq datasets --break-system-packages

## Step 1 - Fetch the on-policy responses

Training data for a speculator has to be **on-policy**: generated by the exact verifier the drafter
will be paired with, so the drafter learns to predict what that model would actually say.
`data-preparation.ipynb` did that generation once and pushed the result
to the Hub, which makes it a stable, versioned input - retraining never re-runs generation, and a
fresh runtime just downloads the file again. (The header says "generate", but nothing is generated
here; this step only downloads and reshapes.)

`local_dir="."` drops the file in the repository root instead of the Hub cache, so the path handed
to `prepare_data.py` later is a plain relative filename. Each row of the JSONL is one conversation
in speculator format - `input_ids` plus a `loss_mask` marking the positions the drafter is scored
on - alongside a `metadata` block carrying the token counts used in the filter below.

Note that `HF_TOKEN` is not set until Step 5, so this download runs unauthenticated. That works for
a public dataset repo; if the repo is private, move the Colab Secrets cell from Step 5 up to here.

In [ ]:
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="yosefw/magpie-llama-3.2-1b-instruct",
    filename="open-perfectblend_Llama-3.2-1B-Instruct.jsonl",
    repo_type="dataset",
    local_dir=".",          # drop it in the cwd instead of the cache
)

print(path)

### Inspect the dataset

Loading the JSONL back through `datasets` gives a row count and column list, and the `print` checks
one row's `metadata.usage` - `total_tokens` and `prompt_tokens` - which is what the next cell
filters on. Worth a glance before committing to a multi-hour run: if `prompt_tokens` looks nothing
like expected, the wrong file was downloaded.

In [ ]:
from datasets import load_dataset

ds = load_dataset(
    "json",
    data_files="open-perfectblend_Llama-3.2-1B-Instruct.jsonl",
    split="train",
)
print(ds)
print(ds[0]['metadata']['usage']['total_tokens'], ds[0]['metadata']['usage']['prompt_tokens'])

### Filter by prompt length

Drops any conversation whose *prompt* exceeds 768 tokens, then writes the survivors back out as
`filtered-open-perfectblend_Llama-3.2-1B-Instruct.jsonl`, which is what Step 2 tokenizes.

The reason is that a speculator only learns from the positions it is scored on - the *response*
tokens. A sample with a 4000-token prompt and a 60-token answer costs a full sequence slot and
contributes almost no training signal. Capping the prompt at 768 against the `--seq-length 1024`
used in Step 2 leaves at least 256 tokens of response in every retained sample.

`num_proc=4` parallelises the scan; `force_ascii=False` keeps non-ASCII text intact rather than
escaping it, which matters since the mix is multilingual.

In [ ]:
ds_filtered = ds.filter(lambda row: row['metadata']['usage']['prompt_tokens'] <= 768, num_proc=4)
ds_filtered.to_json(
    "filtered-open-perfectblend_Llama-3.2-1B-Instruct.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)
ds_filtered

## Step 2 - Tokenize into an Arrow dataset

`prepare_data.py` converts the JSONL into preprocessed Arrow files - fixed-length token sequences
with their loss masks - which is the format the training loop's dataloader reads. This is Step 1 of
the upstream tutorial.

| Flag | Meaning |
| --- | --- |
| `--model` | The verifier, used here for its tokenizer and chat template. |
| `--data` | The **filtered** JSONL from Step 1, not the raw download. |
| `--output ./output` | Destination for the Arrow shards; `train.py` reads this same directory. |
| `--max-samples 100000` | Cap on conversations processed. Drop to a few hundred to smoke-test the pipeline. |
| `--seq-length 1024` | Per-conversation cap; anything longer is truncated. Paired with the 768-token prompt filter above. |
| `--overwrite` | Replaces an existing `./output`, so the cell is safe to re-run. |

**On `--seq-length 1024` vs `--total-seq-len 8192` in Step 4.** These are deliberately different.
`--seq-length` bounds a *single* conversation, while `--total-seq-len` is the training window that
several such conversations are packed into - roughly eight per window here, which keeps the A100
busy without padding waste.

The `rm -r ./output` cell below is a manual reset kept for convenience - `--overwrite` makes it
unnecessary, and it errors on a fresh runtime where `./output` does not exist yet.

In [ ]:
! rm -r ./output

In [ ]:
# in speculators venv
! python scripts/prepare_data.py \
  --model "unsloth/Llama-3.2-1B-Instruct" \
  --data ./filtered-open-perfectblend_Llama-3.2-1B-Instruct.jsonl \
  --output ./output \
  --max-samples 100000 \
  --seq-length 1024 \
  --overwrite

## Step 3 - Serve the verifier for hidden states

`launch_vllm.py` is a thin wrapper around `vllm serve` that turns on hidden-state export, so the
server returns not just tokens but the intermediate activations the drafter is trained to condition
on. It stays up for the whole of Step 4.

| Flag | Why |
| --- | --- |
| `--gpu-memory-utilization 0.25` | **The single-GPU split.** vLLM takes a quarter of the A100 for the verifier's weights and KV cache and leaves the rest to training, which starts on the same device in Step 4. Raising this starves the trainer; lowering it too far shrinks the KV cache until hidden-state requests start queueing. |
| `--max-model-len 8200` | Caps the server's context at just over the 8192-token training window, so the KV cache is sized for what is actually requested rather than the model's full context. |
| `--target-layer-ids 2 8 14` | Which transformer layers get exported. **Must be identical to the list passed to `train.py` in Step 4.** |

There is no cross-check on `--target-layer-ids` between server and trainer - a mismatch means the
drafter asks for layers the server is not exporting, and the run either errors out mid-epoch or
quietly learns from the wrong signal.

`start_new_session=True` puts the server in its own process group, so interrupting a notebook cell
(SIGINT) does not take the server down with it. It is stopped explicitly after training instead.

In [ ]:
import subprocess, json

log_file = open("/content/vllm.log", "w")
process = subprocess.Popen(
    [
        "python", "scripts/launch_vllm.py",
        "unsloth/Llama-3.2-1B-Instruct",
        "--gpu-memory-utilization", "0.25",
        "--max-model-len", "8200",
        "--target-layer-ids", "2", "8", "14",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True,  # detaches from the notebook's process group so a cell interrupt (SIGINT) doesn't reach it
)

print(f"Started vLLM server with PID {process.pid}")

### Wait until the server is ready

Loading the verifier and allocating the KV cache takes minutes. Training would fail immediately
against a server that is not yet listening, so this cell polls `/health` every 5 seconds for up to
about 8 minutes and only returns once vLLM reports ready. If it exhausts the loop without printing
`Server is ready!`, read `/content/vllm.log` - an OOM or a bad model id will be at the bottom of
it.

In [ ]:
import time, requests

for i in range(100):
    try:
        r = requests.get("http://127.0.0.1:8000/health")
        if r.status_code == 200:
            print("Server is ready!")
            break
    except requests.exceptions.ConnectionError:
        pass
    print(f"Waiting... ({i+1})")
    time.sleep(5)

## Step 4 - Train the DSpark drafter

The main event. `CUDA_VISIBLE_DEVICES=0` is the same device vLLM is already on - training takes the
memory the server's `--gpu-memory-utilization 0.25` left behind.
`torchrun --standalone --nproc_per_node 1` runs a single training process.

| Flag | Meaning |
| --- | --- |
| `--verifier-name-or-path` | The model being accelerated. Its config determines the drafter's hidden size and embedding table. |
| `--data-path ./output` | The Arrow dataset from Step 2. |
| `--save-path ./output/checkpoints` | Where per-epoch checkpoints are written. |
| `--draft-vocab-size 32000` | The drafter predicts over a reduced 32K vocabulary instead of the verifier's 128K, which shrinks its output head substantially at a small cost in coverage. |
| `--epochs 3` | Passes over the training split - fewer than the T4 run, since each epoch covers 100K samples. |
| `--train-data-ratio 0.96` | 96/4 train/validation split; validation loss is what picks `checkpoint_best`. |
| `--total-seq-len 8192` | Training window length. Conversations capped at 1024 tokens in Step 2 are packed into it. |
| `--max-anchors 1024` | Cap on the positions per window the drafter is trained from. High here because the A100 has the memory for it; this is one of the main levers on step time. |
| `--num-workers 12` | Dataloader workers, matched to Colab's A100 CPU allocation. |
| `--speculator-type dspark` | Selects the DSpark algorithm (alternatives include `eagle3`, `peagle`, `dflash`, `mtp`). |
| `--num-layers 3` | Depth of the drafter - one more than the 2-layer T4 model. |
| `--block-size 8` | Tokens proposed per verification round. Double the T4 run: a higher ceiling on accepted tokens, but the trailing slots are much harder to get right. |
| `--lr 3e-4` | Peak learning rate. `--optimizer` is omitted, so the trainer's default is used. |
| `--loss-fn '{"ce": 0.1, "tv": 0.9}'` | DSpark's loss mixture: cross-entropy for token prediction plus a total-variation term for confidence calibration, weighted here heavily toward TV. |
| `--scheduler-type cosine` / `--scheduler-warmup-ratio 0.04` | Cosine decay from the peak LR, after a warmup over the first 4% of steps. |
| `--vllm-endpoint` | The server from Step 3. Its presence is what makes this an online run. |
| `--target-layer-ids 2 8 14` | Layers to read hidden states from. **Must match Step 3.** |
| `--on-missing generate` | When a batch's hidden states are absent, request them from the server rather than raising or skipping. |
| `--on-generate delete` | Discard them after the step instead of caching to disk. Together with the flag above, this pair is what defines online mode. |
| `--no-resume-from-checkpoint` | Start fresh rather than picking up an existing checkpoint in `--save-path`. |
| `--log-freq 200` | Steps between log lines. |

`2>&1 | tee /content/train.log` mirrors the output to a file as well as the cell, so the loss curve
survives a browser disconnect - worth having on a run this long.

Output is one directory per epoch under `./output/checkpoints`, each with `config.json`,
`model.safetensors`, and optimizer/scheduler state, plus a `checkpoint_best` symlink pointing at the
lowest validation loss.

In [ ]:
# in speculators venv
! CUDA_VISIBLE_DEVICES=0 torchrun --standalone --nproc_per_node 1 \
  scripts/train.py \
  --verifier-name-or-path "unsloth/Llama-3.2-1B-Instruct" \
  --data-path ./output \
  --save-path ./output/checkpoints \
  --draft-vocab-size 32000 \
  --epochs 3 \
  --train-data-ratio 0.96 \
  --total-seq-len 8192 \
  --max-anchors 1024 \
  --num-workers 12 \
  --speculator-type dspark \
  --num-layers 3 \
  --block-size 8 \
  --lr 3e-4 \
  --loss-fn '{"ce": 0.1, "tv": 0.9}' \
  --scheduler-type cosine \
  --scheduler-warmup-ratio 0.04 \
  --vllm-endpoint http://localhost:8000/v1 \
  --target-layer-ids 2 8 14 \
  --on-missing generate \
  --on-generate delete \
  --no-resume-from-checkpoint \
  --log-freq 200 2>&1 | tee /content/train.log

### Shut down the verifier

`killpg` rather than `kill`, because `start_new_session=True` in Step 3 put the server in its own
process group - signalling only the parent would leave vLLM's worker processes running and holding
its 25% of the GPU. On a single-GPU runtime that memory has to come back before the drafter can be
served in Step 6.

In [ ]:
import os
import signal
os.killpg(os.getpgid(process.pid), signal.SIGTERM)

## Step 5 - Publish the checkpoint

The token is read from Colab Secrets into `HF_TOKEN`, where `huggingface_hub` picks it up
automatically. The secret must be enabled for this notebook (key icon in the sidebar) and needs
**write** scope. Off Colab the `google.colab` import fails; replace it with a direct
`os.environ["HF_TOKEN"] = "hf_..."`.

`checkpoint_best` is self-contained - it carries the drafter weights plus a config recording which
verifier it belongs to - so uploading that directory as-is produces a repository vLLM can serve
directly. `create_repo(..., exist_ok=True)` makes re-runs idempotent.

> **Check the repo ids before running this.** `create_repo` targets
> `yosefw/Llama-3.2-1B-Instruct-DSpark-OPB-3L`, but `upload_folder` pushes to
> `yosefw/Llama-3.2-1B-Instruct-DSpark` - the 2-layer model from the earlier run. As written, this
> creates an empty repo and overwrites a different model. Set both to the same id.

In [ ]:
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get("HF_WRITE_YOSEFW")

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(repo_id="yosefw/Llama-3.2-1B-Instruct-DSpark-OPB-3L", exist_ok=True)

api.upload_folder(
    folder_path="/content/speculators/output/checkpoints/checkpoint_best",
    repo_id="yosefw/Llama-3.2-1B-Instruct-DSpark",
    repo_type="model",
)

## Step 6 - Serve and benchmark

The evaluation harness lives in `scripts/evaluate` with its own requirements. Note the `%cd` - the
remaining cells run from inside that directory, which is two levels below the repository root.

In [ ]:
%cd scripts/evaluate
! pip install -qr requirements.txt

### Serve the trained drafter

Serving the *drafter* checkpoint is enough: its config names the verifier, which vLLM downloads and
loads automatically, then wires the two together for speculative decoding. The drafter is not a
standalone model and will not generate anything on its own. The checkpoint also carries a
`speculators_config` block, so vLLM turns speculative decoding on by itself - no
`--speculative-config` argument is required.

`PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` and `--gpu-memory-utilization 0.80` guard against
fragmentation OOM now that two sets of weights share one card. As in Step 3, the server is detached
with `start_new_session=True` and the cell blocks on a `/health` poll until ready - up to 10 minutes,
since the verifier is fetched from the Hub on first run.

> **The model path is relative to the wrong directory.** `./output/checkpoints/checkpoint_best`
> resolves from `scripts/evaluate` after the `%cd` above, where it does not exist. Use the absolute
> path `/content/speculators/output/checkpoints/checkpoint_best`, or the Hub repo id once Step 5 has
> pushed it.

In [ ]:
import subprocess, os

env = os.environ.copy()
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

log_file = open("/content/vllm_serve.log", "w")
process = subprocess.Popen(
    [
        "vllm", "serve", "./output/checkpoints/checkpoint_best",
        "--port", "8000",
        "--gpu-memory-utilization", "0.80"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=env,
    start_new_session=True,  # detaches from notebook's process group so cell interrupts don't kill it
)
print(f"Started vLLM server with PID {process.pid}")

import time, requests

for i in range(120):
    try:
        r = requests.get("http://127.0.0.1:8000/health")
        if r.status_code == 200:
            print("Server is ready!")
            break
    except requests.exceptions.ConnectionError:
        pass
    print(f"Waiting... ({i+1})")
    time.sleep(5)

### Measure acceptance

`evaluate.py throughput` replays a fixed benchmark suite through the endpoint and reports, per
subset:

- **`acceptance_length`** - the mean number of tokens kept per verification round. With
  `--block-size 8` the ceiling is 9 (the proposed block plus the verifier's own token) and 1.0 means
  nothing was ever accepted. This is an *upper bound* on single-stream speedup: it does not charge
  for the drafter's own forward pass, which is larger here than in the 2-layer T4 model.
- **`pos_0` .. `pos_7`** - the acceptance rate at each slot of the proposed block. These decay across
  the block by construction, since a slot can only be accepted if every slot before it was. They use
  a different denominator from `acceptance_length`, so they do not sum to it.

With a block of 8 the tail slots are where this configuration either pays for itself or does not:
if `pos_4` onward collapse to near zero, the extra block length is costing drafter compute for
nothing, and a smaller `--block-size` would serve better.

Acceptance is highest on structured, predictable text - code, math, tool calls - and lowest on
translation and summarization.

In [ ]:
! python evaluate.py throughput --target http://localhost:8000/v1

### Shut down the evaluation server

Frees the GPU again. Same `killpg` reasoning as after training - the serving process group has
workers of its own that a plain `kill` would leave behind.

In [ ]:
import os
import signal
os.killpg(os.getpgid(process.pid), signal.SIGTERM)

## Results and next steps

The published [README](../README.md) numbers - **2.404** weighted acceptance length and **2.17x**
wall-clock throughput - belong to the **2-layer, block-4** drafter trained on 2x T4, not to this
configuration. Treat them as the baseline this run is trying to beat, and record whatever
`evaluate.py` prints above against it.

Two things to compare when it finishes:

- **Acceptance length** should rise, since the block is twice as long and the drafter a layer
  deeper. If it does not clear 2.404, the extra capacity is not paying off.
- **Wall-clock ratio** is the honest test. A 3-layer drafter costs more per proposal than a 2-layer
  one, so acceptance has to rise enough to cover that. It is entirely possible for acceptance to go
  up while throughput goes down.

Serve the finished drafter anywhere vLLM runs:

```bash
vllm serve yosefw/Llama-3.2-1B-Instruct-DSpark-OPB-3L \
  --port 8000 \
  --gpu-memory-utilization 0.80
```

Then query the OpenAI-compatible endpoint at `http://localhost:8000/v1` as usual - speculative
decoding is transparent to the caller.